<a href="https://colab.research.google.com/github/aims-ai-research-foundations/pilot-workshop/blob/main/assignments/day2/day2-course3-student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Long Activity 2 — Lab: Neural Network Training
## Training a Small Neural Network and Observing Overfitting

**Student Notebook**

In this lab you will train a small neural network on a noisy 2-D classification problem and **see overfitting with your own eyes**. You'll then design and run two experiments to fight that overfitting.

**Time:** 60 minutes.
**Linked online activity:** *Lab: Mitigate Overfitting*

### Learning objectives
By the end of this lab you should be able to:
- Explain overfitting and generalisation conceptually.
- Interpret training vs. test accuracy on the same model.
- Relate model complexity (and regularisation) to bias and variance.

### How this notebook is organised
1. **Setup & data** (provided) — run these cells, look at the picture.
2. **Baseline BIG model** (provided) — read this carefully. You'll need it as a template.
3. **Experiment A — reduce capacity** (you write the code).
4. **Experiment B — add regularisation** (you write the code).
5. **Compare & reflect** (provided plot + your conclusions).


## Step 1 — Imports and fixed seed *(provided — just run)*
The seed makes every run reproducible. Your group's numbers should match every other group's, so we can compare like-for-like.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score

SEED = 7
np.random.seed(SEED)
print('Setup done — seed fixed at', SEED)

Setup done — seed fixed at 7


## Step 2 — Generate a noisy 2-D dataset *(provided)*
We deliberately make the dataset **small (100 points) and noisy (noise = 0.40)**. With this much noise, a model that tries too hard to fit every training point will pay the price on unseen data.

In [ ]:
X, y = make_moons(n_samples=100, noise=0.40, random_state=SEED)

plt.figure(figsize=(6, 5))
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=60)
plt.title('Our dataset (colour is the label we want to predict)')
plt.xlabel('x1'); plt.ylabel('x2')
plt.show()
print(f'Dataset shape: {X.shape},  class balance: {(y == 0).sum()}/{(y == 1).sum()}')

## Step 3 — Train / test split *(provided)*
70 % for training, 30 % held out as a test set. **The test set is never seen during training.** Test accuracy is our honest estimate of how the model will behave on new data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=SEED
)
print(f'Training points: {len(X_train)},  Test points: {len(X_test)}')

Training points: 70,  Test points: 30


## Step 4 — Decision-boundary helper *(provided — just run)*
This function draws the model's **decision boundary** — the line where it switches between predicting blue and red — over the data. A model that has learned the real pattern draws a smooth curve; a model that is overfitting draws a wiggly boundary chasing noise. **You will call this helper after training each model.**

In [ ]:
def plot_decision_boundary(model, X, y, title='', ax=None):
    '''Plot the decision boundary of a fitted model over the data points.'''
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 5))
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='coolwarm')
    ax.scatter(X[:, 0], X[:, 1], c=y, cmap='coolwarm', edgecolor='k', s=50)
    ax.set_title(title)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    return ax

print('Helper ready. Call: plot_decision_boundary(model, X_train, y_train, title=...)')

Helper ready. Call: plot_decision_boundary(model, X_train, y_train, title=...)


## Step 5 — Baseline BIG model *(worked example — read carefully)*

We give you the **first** model in full. Read it. **You will use this as your template for Experiments A and B**, so make sure you understand every line.

**Predict before you run:** Do you expect this model to overfit or underfit this 100-point noisy dataset? Why?

In [ ]:
# ===== WORKED EXAMPLE — do not modify =====
# Your prediction (write your reasoning here before running):
# I expect this model to ___________ because ___________ .

# 1) Define the model.
#    hidden_layer_sizes=(100, 100) means two hidden layers of 100 neurons each.
#    max_iter=3000 gives the optimiser plenty of iterations to converge.
#    random_state=SEED makes the result reproducible.
model_big = MLPClassifier(
    hidden_layer_sizes=(100, 100),
    max_iter=3000,
    random_state=SEED,
)

# 2) Fit (= train) the model on the TRAINING data only.
model_big.fit(X_train, y_train)

# 3) Compute accuracy on both sets.
#    accuracy_score(y_true, y_predicted) — we predict, then compare.
train_acc_big = accuracy_score(y_train, model_big.predict(X_train))
test_acc_big  = accuracy_score(y_test,  model_big.predict(X_test))

# 4) Report.
print(f'BIG model  →  train acc: {train_acc_big:.3f}   test acc: {test_acc_big:.3f}')
print(f'             gap (train − test): {train_acc_big - test_acc_big:+.3f}')

# 5) Visualise the decision boundary.
plot_decision_boundary(model_big, X_train, y_train,
                       title=f'BIG model — train {train_acc_big:.2f} / test {test_acc_big:.2f}')
plt.show()

**Reflection:**
- Look at the boundary. Where does it wiggle? Why is it bending around individual points?
- Look at the numbers. How large is the gap between training and test accuracy? Is that big or small? What does the size of the gap tell you?
- Are these results consistent with your prediction above?

## Step 6 — Experiment A: shrink the network *(you implement)*

If the BIG model is **too complex** for this small dataset, a smaller network might generalise better. **Build one and find out.**

### What you need to do
Train a *smaller* neural network on the **same** training data and compute its train/test accuracy.

### Hints
1. The class is `MLPClassifier` — it's already imported.
2. Look at the BIG-model code above as a template. **Most lines stay the same**: you only change the model's *architecture*.
3. Try a much smaller `hidden_layer_sizes` — for example `(8,)`, `(16,)`, or `(4, 4)`. **Start with one, run the cell, then try another.**
4. Keep `max_iter=3000` and `random_state=SEED`. (Why? See Step 1.)
5. Use the same `accuracy_score(...)` pattern as in the BIG example.
6. The reporting/plot block at the bottom of the cell already references the variables you must define: `your_hidden_layers`, `model_small`, `train_acc_small`, `test_acc_small`. **Use those exact names** or the reporting block won't work.


In [ ]:
# ===== EXPERIMENT A: you implement =====

your_hidden_layers = ...   # ← record your choice as a tuple

# TODO: build the smaller model.
model_small = ...

# TODO: fit your model on (X_train, y_train).

# TODO: compute training and test accuracy using accuracy_score.
train_acc_small = ...
test_acc_small  = ...

# ----- Reporting & plot (do not modify) -----
print(f'SMALL model {your_hidden_layers}')
print(f'  train acc: {train_acc_small:.3f}   test acc: {test_acc_small:.3f}')
print(f'  gap (train − test): {train_acc_small - test_acc_small:+.3f}')

plot_decision_boundary(model_small, X_train, y_train,
                       title=f'SMALL {your_hidden_layers} — train {train_acc_small:.2f} / test {test_acc_small:.2f}')
plt.show()

**Compare with the BIG model:** What happened to training accuracy? What happened to test accuracy? Did the boundary get smoother? Try at least one more `hidden_layer_sizes` value before moving on — does the test accuracy keep improving, or does it eventually drop?

## Step 7 — Experiment B: keep the big network, but regularise it *(you implement)*

Shrinking the network isn't the only way to fight overfitting. Another way is to **keep the high capacity but penalise complex solutions** — this is called **regularisation**.

### What you need to do
Train a neural network with **the same architecture as the BIG model** (100, 100), but turn regularisation up.

### Hints
1. In scikit-learn's `MLPClassifier`, L2 regularisation strength is controlled by the **`alpha`** parameter. (Look at the docstring: `?MLPClassifier`.)
2. The default is `alpha=0.0001` — essentially "off". **You want a much larger value.**
3. Suggested values to try (left → right = stronger regularisation): `0.01`, `0.1`, `1.0`, `10.0`.
4. **Important:** keep `hidden_layer_sizes=(100, 100)` — the same as BIG. The whole point is to demonstrate regularisation, not a smaller network.
5. The reporting block expects `your_alpha`, `model_reg`, `train_acc_reg`, `test_acc_reg`. Use those names.


In [ ]:
# ===== EXPERIMENT B: you implement =====

your_alpha = ...   # ← record your choice as a float

# TODO: build the regularised model.
model_reg = ...

# TODO: fit your model on (X_train, y_train).

# TODO: compute training and test accuracy using accuracy_score.
train_acc_reg = ...
test_acc_reg  = ...

# ----- Reporting & plot (do not modify) -----
print(f'REGULARISED model (alpha={your_alpha})')
print(f'  train acc: {train_acc_reg:.3f}   test acc: {test_acc_reg:.3f}')
print(f'  gap (train − test): {train_acc_reg - test_acc_reg:+.3f}')

plot_decision_boundary(model_reg, X_train, y_train,
                       title=f'REG α={your_alpha} — train {train_acc_reg:.2f} / test {test_acc_reg:.2f}')
plt.show()

**Compare with the BIG model:** Same architecture, different alpha. Did the boundary get smoother? Did the train-vs-test gap shrink? Try at least two alpha values — when you crank it too high, what happens to *both* accuracies?

## Step 8 — Side-by-side comparison *(provided — just run after Steps 5–7 succeed)*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
plot_decision_boundary(model_big,   X_train, y_train,
    f'BIG (100,100)\ntrain {train_acc_big:.2f}  test {test_acc_big:.2f}', axes[0])
plot_decision_boundary(model_small, X_train, y_train,
    f'SMALL {your_hidden_layers}\ntrain {train_acc_small:.2f}  test {test_acc_small:.2f}', axes[1])
plot_decision_boundary(model_reg,   X_train, y_train,
    f'REG α={your_alpha}\ntrain {train_acc_reg:.2f}  test {test_acc_reg:.2f}', axes[2])
plt.tight_layout()
plt.show()

print('\nSummary table')
print('=' * 56)
print(f"{'Model':<22} {'train':>8} {'test':>8} {'gap':>8}")
print('-' * 56)
print(f"{'BIG (100,100)':<22} {train_acc_big:>8.3f} {test_acc_big:>8.3f} {train_acc_big - test_acc_big:>+8.3f}")
print(f"{f'SMALL {your_hidden_layers}':<22} {train_acc_small:>8.3f} {test_acc_small:>8.3f} {train_acc_small - test_acc_small:>+8.3f}")
print(f"{f'REG alpha={your_alpha}':<22} {train_acc_reg:>8.3f} {test_acc_reg:>8.3f} {train_acc_reg - test_acc_reg:>+8.3f}")

## Step 9 — Reflection

1. Which model had the **highest training accuracy**? Which had the **highest test accuracy**? Why aren't those the same model?
2. Point to ONE specific place on the BIG-model boundary where you can see it learning **noise** rather than signal. Describe what you see.
3. Your two experiments (smaller network, stronger regularisation) attack overfitting from different angles. **Did they give similar test accuracy, or did one clearly win?**
4. If you had to deploy one of these three models in the real world, which would you choose — and why?
5. **Bias–variance link:** which model has high bias? Which has high variance? Which one sits closest to a good trade-off?


**Your answers:**

1.

2.

3.

4.

5.



---
### If you finish early
- In Experiment A, sweep through several sizes and **record the test accuracy for each**. Does it keep improving, or does it eventually drop (the model gets too simple = underfitting)? Plot the result.
- In Experiment B, find the **best** alpha. Then ask: how could you have chosen this value **without** peeking at the test set? (Hint: validation set.)
- Try changing the dataset noise (`noise=0.6`) and re-running. Does the gap between train and test get larger or smaller for the BIG model? Why?